# Views and changing data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-12-views-and-changing-data.ipynb)

Queries read data. Production pipelines also create objects and change data: they define views, create tables,
add rows, replace partitions, and merge new records into existing tables. This chapter covers the statements
for each, and which ones need a table format that supports transactions.

Spark runs in this notebook with its built-in tables. Statements that need Delta Lake or Apache Iceberg are shown
as reference code, because those formats are not part of this notebook's setup.

## Setup

Install PySpark and start a Spark session. Tables go to a temporary warehouse directory, so no files are left in the notebook folder.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.warehouse.dir", "/tmp/sql-learning-warehouse")
    .getOrCreate()
)

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

The `orders` temporary view has 14 rows: 11 `completed`, 2 `cancelled` and 1 `pending`, across six customers and
three months (January to March 2023). The completed amounts add up to 2995. Every statement below starts from this
view, so you can check each result against these numbers.

In [ ]:
from pyspark.errors import AnalysisException, UnsupportedOperationException

q = "SELECT status, COUNT(*) AS n, SUM(amount) AS total FROM orders GROUP BY status ORDER BY status"
spark.sql(q).show()
assert spark.sql(q).collect() == [("cancelled", 2, 575), ("completed", 11, 2995), ("pending", 1, 220)]

## 1. Views

A **view** is a saved query with a name. It stores no rows. When you query a view, the engine substitutes the saved
query text and runs it against the tables underneath, so the result always reflects the current data.

There are two kinds:

- A **temporary view** belongs to the current Spark session and disappears when the session ends. The `orders`
  view in the setup is one.
- A **permanent view** is stored in the **catalog**, the database of table and view definitions that every
  session can read. Other sessions, and other people, can query it.

### Where you'll use it

- **Stable interfaces:** a view named `active_customers` lets analysts query one definition, and the definition changes in one place.
- **Access control:** a view that hides personal columns from a reporting role, while the role has no access to the table.
- **Renaming and reshaping:** a view keeps an old column name working while the table underneath is migrated.

### A permanent view cannot read a temporary view

A permanent view outlives the session, but a temporary view does not. If a permanent view could read `orders`, the
next session would find a view whose source no longer exists. Spark therefore rejects the statement. The cell below
proves it.

In [ ]:
spark.sql("DROP VIEW IF EXISTS completed_orders")
try:
    spark.sql("CREATE VIEW completed_orders AS SELECT * FROM orders WHERE status = 'completed'")
    rejected = False
except AnalysisException as err:
    rejected = "INVALID_TEMP_OBJ_REFERENCE" in str(err)
assert rejected

So the orders are first copied into a real table, `orders_table`, and the permanent view `completed_orders` reads
that table. The view keeps three columns and only the completed rows.

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_table")
spark.sql("CREATE TABLE orders_table USING PARQUET AS SELECT * FROM orders")
spark.sql("""
CREATE VIEW completed_orders AS
SELECT order_id, customer_id, amount
FROM orders_table
WHERE status = 'completed'
""")

q = "SELECT COUNT(*) AS completed, SUM(amount) AS total FROM completed_orders"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11, 2995)]

Views keep the logic in one place. If the rule changes, you change the view, and every query that reads it picks up
the change. Views do not make a query faster by themselves, because the engine still runs the saved query on every
read. Some warehouses offer a **materialised view**, which stores the result and refreshes it; that is a table
managed for you, not a plain view.

## 2. Creating tables from a query

`CREATE TABLE ... AS SELECT` (often shortened to **CTAS**) creates a table and fills it in one statement. The column
names and types come from the query. Unlike a view, the table stores the rows as files at the moment the statement
runs. It is a **snapshot**: later changes to the source do not reach it.

### Where you'll use it

- **Materialised outputs:** CTAS is how a pipeline writes a snapshot, a backfill or a reporting table.
- **Debugging:** freezing the output of a step in a table, so you can query it repeatedly without re-running the step.

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_snapshot")
spark.sql("""
CREATE TABLE completed_snapshot USING PARQUET AS
SELECT * FROM completed_orders
""")

q = "SELECT COUNT(*) AS rows_in_snapshot FROM completed_snapshot"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11,)]

### View versus snapshot, worked by hand

Add one completed order (id 15, amount 500) to `orders_table`. Then:

| Object | What a read does | Rows after the insert |
| --- | --- | --- |
| `completed_orders` (view) | re-runs the query on `orders_table` | 12 |
| `completed_snapshot` (table) | reads the files written by CTAS | 11 |

The cell below makes the insert and checks both counts. It then resets `orders_table` to the original 14 rows with
`INSERT OVERWRITE`, which section 4 explains.

In [ ]:
spark.sql("INSERT INTO orders_table VALUES (15, 1, 'completed', 500, DATE '2023-03-31')")
q = """
SELECT (SELECT COUNT(*) FROM completed_orders)   AS view_rows,
       (SELECT COUNT(*) FROM completed_snapshot) AS snapshot_rows
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(12, 11)]

spark.sql("INSERT OVERWRITE TABLE orders_table SELECT * FROM orders")
assert spark.sql("SELECT COUNT(*) FROM completed_orders").collect() == [(11,)]

## 3. Adding rows: INSERT INTO

`INSERT INTO ... VALUES` adds literal rows. `INSERT INTO ... SELECT` adds the rows of a query, which is how most
pipelines **append** a new batch: the existing rows stay, and the new rows are added after them.

Without a column list, values are matched to columns **by position**, not by name. `completed_snapshot` has the
columns `(order_id, customer_id, amount)`, so `VALUES (15, 1, 500)` means order 15, customer 1, amount 500. Writing
the column list, as in the second statement below, makes the mapping explicit and survives a change in column order.

### Where you'll use it

- **Appending daily loads:** a scheduled job adds yesterday's rows to a fact table.
- **Event logs:** audit and event tables only ever grow, so every write is an append.

In [ ]:
spark.sql("INSERT INTO completed_snapshot VALUES (15, 1, 500)")
spark.sql("""
INSERT INTO completed_snapshot (order_id, customer_id, amount)
SELECT order_id, customer_id, amount FROM completed_orders WHERE order_id = 1
""")

q = "SELECT COUNT(*) AS rows_after_inserts FROM completed_snapshot"
spark.sql(q).show()
assert spark.sql(q).collect() == [(13,)]

### Appending is not idempotent

An operation is **idempotent** when running it twice leaves the same result as running it once. Appending is not.
The snapshot already held order 1, and the second insert added it again, so order 1 now appears twice. A job that
fails halfway and is re-run with `INSERT INTO` duplicates every row it had already written.

In [ ]:
q = "SELECT order_id, COUNT(*) AS copies FROM completed_snapshot GROUP BY order_id HAVING COUNT(*) > 1"
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 2)]

## 4. Replacing data: INSERT OVERWRITE

`INSERT OVERWRITE` removes the existing rows and writes the result of the query in their place. Running it again
with the same input produces the same table, so it is idempotent. It is the simplest way to make a load safe to re-run.

Below, the snapshot (13 rows, with order 1 duplicated) is replaced by the orders with an amount of 300 or more. By
hand, those are orders 4 (300), 6 (400), 10 (320), 11 (500), 12 (450) and 14 (325): six rows. The statement runs twice
to show that the second run changes nothing.

### Where you'll use it

- **Idempotent daily partitions:** overwriting one partition makes a rerun produce the same result, which is why Spark and Hive pipelines use it.
- **Full refreshes:** small dimension tables are often rebuilt completely on every run.

In [ ]:
overwrite = """
INSERT OVERWRITE TABLE completed_snapshot
SELECT order_id, customer_id, amount FROM orders WHERE amount >= 300
"""
q = "SELECT order_id FROM completed_snapshot ORDER BY order_id"
for _ in range(2):
    spark.sql(overwrite)
    assert spark.sql(q).collect() == [(4,), (6,), (10,), (11,), (12,), (14,)]
spark.sql("SELECT COUNT(*) AS rows_after_overwrite FROM completed_snapshot").show()

### A table cannot overwrite itself

Spark refuses an `INSERT OVERWRITE` whose query reads the same table it writes. It would delete the files it is
still reading. Write the result to a new table first, or read from the source the table was built from.

In [ ]:
try:
    spark.sql("INSERT OVERWRITE TABLE completed_snapshot SELECT * FROM completed_snapshot WHERE amount > 400")
    rejected = False
except AnalysisException as err:
    rejected = "UNSUPPORTED_OVERWRITE" in str(err)
assert rejected

### Overwriting one partition

A **partitioned table** stores its rows in one directory per value of a **partition column**, such as
`order_month=2023-01/`. Overwriting a partition replaces one directory and leaves the others alone, so a daily job
can rewrite one day without touching the rest.

The table below holds the 14 orders partitioned by month: 4 rows in January, 5 in February and 5 in March.

In [ ]:
spark.sql("DROP TABLE IF EXISTS orders_by_month")
spark.sql("""
CREATE TABLE orders_by_month USING PARQUET PARTITIONED BY (order_month) AS
SELECT order_id, customer_id, status, amount, date_format(order_date, 'yyyy-MM') AS order_month
FROM orders
""")

MONTH_COUNTS = "SELECT order_month, COUNT(*) AS n FROM orders_by_month GROUP BY order_month ORDER BY order_month"
spark.sql(MONTH_COUNTS).show()
assert spark.sql(MONTH_COUNTS).collect() == [("2023-01", 4), ("2023-02", 5), ("2023-03", 5)]

Name the partition in a `PARTITION (...)` clause to overwrite only that partition. Here March is replaced by its
completed orders only. March had five orders, one of them cancelled (order 14), so March drops to four rows and the
other months keep theirs. The query no longer selects `order_month`, because the clause supplies it.

In [ ]:
spark.sql("""
INSERT OVERWRITE TABLE orders_by_month PARTITION (order_month = '2023-03')
SELECT order_id, customer_id, status, amount
FROM orders
WHERE date_format(order_date, 'yyyy-MM') = '2023-03' AND status = 'completed'
""")
spark.sql(MONTH_COUNTS).show()
assert spark.sql(MONTH_COUNTS).collect() == [("2023-01", 4), ("2023-02", 5), ("2023-03", 4)]

### The static-mode trap

Without a `PARTITION` clause, what gets replaced depends on the setting `spark.sql.sources.partitionOverwriteMode`:

- `static`, the default, replaces **the whole table**, even if the query only produces rows for one partition.
- `dynamic` replaces only the partitions that appear in the query's result.

The cell below writes February's rows without a `PARTITION` clause, first in static mode and then in dynamic mode.
In static mode January and March are deleted. The table is rebuilt between the two runs.

In [ ]:
february = """
INSERT OVERWRITE TABLE orders_by_month
SELECT order_id, customer_id, status, amount, date_format(order_date, 'yyyy-MM') AS order_month
FROM orders
WHERE date_format(order_date, 'yyyy-MM') = '2023-02'
"""
everything = february.replace("WHERE date_format(order_date, 'yyyy-MM') = '2023-02'", "")

assert spark.conf.get("spark.sql.sources.partitionOverwriteMode").lower() == "static"
spark.sql(february)
assert spark.sql(MONTH_COUNTS).collect() == [("2023-02", 5)]

spark.sql(everything)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.sql(february)
spark.sql(MONTH_COUNTS).show()
assert spark.sql(MONTH_COUNTS).collect() == [("2023-01", 4), ("2023-02", 5), ("2023-03", 5)]
spark.conf.unset("spark.sql.sources.partitionOverwriteMode")

Pipelines that rewrite partitions either set dynamic mode or name the partition explicitly. Relying on the default
is how a backfill of one day empties a table.

## 5. UPDATE, DELETE and MERGE

`UPDATE` changes column values in the rows that match a condition. `DELETE` removes the rows that match a condition.
`MERGE` compares a **source** (the new data) with a **target** (the table) on a key, and decides per row whether to
update, insert or delete. Inserting new keys and updating existing ones in one statement is called an **upsert**.

These statements change individual rows, so they need a table format that records each change in a transaction log,
such as **Delta Lake**, **Apache Iceberg** or **Apache Hudi**, or a cloud warehouse table. Plain Parquet files are
written once and never edited in place.

### Where you'll use it

- **Applying changes:** a `MERGE` applies a CDC batch of inserts, updates and deletes to a warehouse or lakehouse table.
- **Privacy requests:** deleting one customer's rows to meet a GDPR erasure request.
- **Corrections:** an `UPDATE` that fixes a mis-mapped status code in historical rows.

Spark's built-in Parquet tables and temporary views reject all three statements. The cell below proves it.

In [ ]:
rejected = []
for stmt in [
    "UPDATE completed_snapshot SET amount = amount + 10 WHERE order_id = 4",
    "DELETE FROM completed_snapshot WHERE amount < 350",
    "MERGE INTO completed_snapshot t USING completed_orders s ON t.order_id = s.order_id "
    "WHEN MATCHED THEN UPDATE SET t.amount = s.amount",
]:
    try:
        spark.sql(stmt)
    except (AnalysisException, UnsupportedOperationException):
        rejected.append(stmt.split()[0])
assert rejected == ["UPDATE", "DELETE", "MERGE"]

### The statements on a table that supports them

Start with `accounts` holding `(1, 50)`, `(2, 70)` and `(3, 40)` as `(id, balance)`, and a source
`account_updates` holding `(1, 65)` and `(4, 90)`. The comments show the table after each statement.

```sql
UPDATE accounts SET balance = balance + 10 WHERE id = 1;
-- (1, 60), (2, 70), (3, 40)

DELETE FROM accounts WHERE balance < 60;
-- (1, 60), (2, 70)              id 3 had 40; id 1 has 60, which is not < 60

MERGE INTO accounts AS target
USING account_updates AS source
  ON target.id = source.id
WHEN MATCHED THEN UPDATE SET target.balance = source.balance          -- id 1 becomes 65
WHEN NOT MATCHED THEN INSERT (id, balance) VALUES (source.id, source.balance)   -- id 4 is added
WHEN NOT MATCHED BY SOURCE THEN DELETE;                               -- id 2 is not in the source
-- (1, 65), (4, 90)
```

Without the last clause, the `MERGE` is a plain upsert and the result is `(1, 65)`, `(2, 70)`, `(4, 90)`.
`WHEN NOT MATCHED BY SOURCE` is supported by Delta Lake, Databricks and SQL Server; check your engine before using it.

### What MERGE computes, as a query

A `MERGE` is a join between target and source followed by a rewrite. The upsert above is a `FULL OUTER JOIN` on the
key: matched keys take the source value, target-only keys keep their value, and source-only keys are inserted.
`COALESCE(source, target)` expresses exactly that. Running this query and then overwriting the table is how an
upsert is done on a format without `MERGE`.

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW accounts AS SELECT * FROM VALUES (1, 60), (2, 70) AS accounts(id, balance)")
spark.sql(
    "CREATE OR REPLACE TEMP VIEW account_updates AS SELECT * FROM VALUES (1, 65), (4, 90) AS account_updates(id, balance)"
)
q = """
SELECT COALESCE(s.id, t.id) AS id, COALESCE(s.balance, t.balance) AS balance
FROM accounts t
FULL OUTER JOIN account_updates s ON t.id = s.id
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 65), (2, 70), (4, 90)]

### Deduplicate the source first

If two source rows share a key, they both match the same target row, and the engine cannot tell which value should
win. Most engines refuse such a `MERGE`. The join below shows the cause: a source with two rows for id 1 produces two
candidate rows for one target row. The deduplication chapter shows how to keep one row per key.

In [ ]:
q = """
SELECT t.id, COUNT(*) AS matching_source_rows
FROM accounts t
JOIN (SELECT * FROM VALUES (1, 65), (1, 80) AS s(id, balance)) s ON t.id = s.id
GROUP BY t.id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 2)]

## 6. Transactions

A **transaction** is a group of statements that takes effect as one unit: either all of them are applied, or none
are. A bank transfer is the standard example. It debits one account and credits another, and a crash between the two
must not leave money missing.

```sql
BEGIN;
UPDATE accounts SET balance = balance - 20 WHERE id = 1;
UPDATE accounts SET balance = balance + 20 WHERE id = 2;
COMMIT;   -- or ROLLBACK; to undo both
```

Table formats such as Delta and Iceberg commit each write **atomically**: a write becomes visible through one entry in
the transaction log, so a reader sees the old table or the new one, never a partial write. Spark's built-in tables and
temporary views give no such guarantee. The interview concepts chapter explains ACID and isolation levels.

### Where you'll use it

- **All-or-nothing loads:** a multi-table update in an OLTP database, or a warehouse transaction that must not show readers half of a change.
- **Concurrent writers:** two jobs writing the same Delta or Iceberg table; the log makes one commit win and the other retry.

## Interview notes

- **Views vs tables.** A view is a saved query, recomputed on every read; a table stores rows. Use a table when
  the query is expensive or many queries read the same result.
- **Idempotent loads.** Use `INSERT OVERWRITE` for a partition or a whole table, or `MERGE` on a business key.
  Either one gives the same result when the job runs twice. `INSERT INTO` does not.
- **Static vs dynamic partition overwrite.** In Spark's default static mode, `INSERT OVERWRITE` without a
  `PARTITION` clause replaces the whole table. Set dynamic mode or name the partition.
- **MERGE needs a unique source key.** Deduplicate the source first, or the statement fails or misbehaves.
- **Hard delete vs soft delete.** A soft delete sets a flag or an end date and keeps the row, so history survives.
  A hard delete removes the row. Analytics tables usually keep history.

## Practice questions

1. Create a table `completed_check` from the `completed_orders` view. How many rows does it have?
2. Append one row to `completed_check` with `order_id` 16, `customer_id` 2 and amount 400. How many rows now?
3. Using `INSERT OVERWRITE`, replace `completed_check` with the orders whose amount is 450 or more. How many rows?
4. In `orders_by_month`, replace only the January partition with January's orders of 200 or more. Return the row
   count per month.
5. Without `MERGE`, compute the upsert of `account_updates` into `accounts` when the source is
   `(2, 75)` and `(5, 10)`. Return `(id, balance)` sorted by id.

## Answers

### Question 1

CTAS copies the view's current result: the 11 completed orders.

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_check")
spark.sql("CREATE TABLE completed_check USING PARQUET AS SELECT * FROM completed_orders")
q = "SELECT COUNT(*) AS n FROM completed_check"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11,)]

### Question 2

`INSERT INTO` appends, so the count grows by one.

In [ ]:
spark.sql("INSERT INTO completed_check VALUES (16, 2, 400)")
q = "SELECT COUNT(*) AS n FROM completed_check"
spark.sql(q).show()
assert spark.sql(q).collect() == [(12,)]

### Question 3

`INSERT OVERWRITE` discards all 12 rows. Only orders 11 (500) and 12 (450) reach 450, so two rows remain.

In [ ]:
spark.sql(
    "INSERT OVERWRITE TABLE completed_check SELECT order_id, customer_id, amount FROM orders WHERE amount >= 450"
)
q = "SELECT order_id FROM completed_check ORDER BY order_id"
spark.sql(q).show()
assert spark.sql(q).collect() == [(11,), (12,)]

### Question 4

Name the partition so the other months survive. January's orders are 1 (100), 4 (300), 7 (120) and 11 (500); two of
them reach 200, so January drops to two rows. February and March keep 5 rows each.

In [ ]:
spark.sql("""
INSERT OVERWRITE TABLE orders_by_month PARTITION (order_month = '2023-01')
SELECT order_id, customer_id, status, amount
FROM orders
WHERE date_format(order_date, 'yyyy-MM') = '2023-01' AND amount >= 200
""")
spark.sql(MONTH_COUNTS).show()
assert spark.sql(MONTH_COUNTS).collect() == [("2023-01", 2), ("2023-02", 5), ("2023-03", 5)]

### Question 5

A `FULL OUTER JOIN` on `id`, preferring the source value. Id 2 matches and becomes 75, id 1 is target-only and keeps
60, and id 5 is source-only and is inserted.

In [ ]:
q = """
SELECT COALESCE(s.id, t.id) AS id, COALESCE(s.balance, t.balance) AS balance
FROM accounts t
FULL OUTER JOIN (SELECT * FROM VALUES (2, 75), (5, 10) AS s(id, balance)) s ON t.id = s.id
ORDER BY id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [(1, 60), (2, 75), (5, 10)]

In [ ]:
spark.sql("DROP TABLE IF EXISTS completed_check")
spark.sql("DROP TABLE IF EXISTS completed_snapshot")
spark.sql("DROP TABLE IF EXISTS orders_by_month")
spark.sql("DROP VIEW IF EXISTS completed_orders")
spark.sql("DROP TABLE IF EXISTS orders_table")